In [1]:
!git clone https://github.com/WebNLG/GenerationEval.git

Cloning into 'GenerationEval'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 109 (delta 49), reused 88 (delta 34), pack-reused 0
Receiving objects: 100% (109/109), 1.21 MiB | 1.39 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [2]:
import os
os.chdir('GenerationEval')

In [3]:
!git checkout humaneval

Branch 'humaneval' set up to track remote branch 'humaneval' from 'origin'.
Switched to a new branch 'humaneval'


In [103]:
import copy
import json
import numpy as np
import os
import pandas as pd
from scipy.stats import zscore
from statsmodels.stats.inter_rater import fleiss_kappa
from scipy.stats import ranksums

SYSTEMS_PATH = 'data/human_evaluation/ratings/'
REFERENCES_PATH = 'data/human_evaluation/data_REF.json'
DOMAIN_PATH = 'data/human_evaluation/ref2domain.json'

# Parsing the data

In [62]:
rdfs = json.load(open(REFERENCES_PATH))
sys_files = [w for w in os.listdir(SYSTEMS_PATH) if not w.startswith('.')]
domains = json.load(open(DOMAIN_PATH))

doc_id = 1
data = []
for sys_file in sys_files:
    results = json.load(open(os.path.join(SYSTEMS_PATH, sys_file)))
    submission_id = sys_file.replace('.json', '').replace('human_eval_id_', '')

    for sample_id in results:
        entry = [w for w in rdfs['entries'] if list(w.keys())[0] == sample_id][0]
        for worker_id in results[sample_id]:
            assign = results[sample_id][worker_id]
            inp = {
                'id': doc_id,
                'sample_id': sample_id,
                'domain': domains['Id' + str(sample_id)],
                'submission_id': submission_id,
                'worker_id': worker_id,
                'category': entry[sample_id]['category'],
                'size': entry[sample_id]['size'],
            }
            inp.update(assign)
            data.append(inp)
            doc_id += 1

# Inter-rater Agreement
## Fleiss' Kappa

Discretize the ratings in 5 categories

In [63]:
n_cat, max_range = 5, 100 # number of categories
data_discretized = []

ids = [w['id'] for w in data]
correctness = [int((n_cat* w['Correctness']) / (max_range+1)) for w in data]
coverage = [int((n_cat* w['DataCoverage']) / (max_range+1)) for w in data]
fluency = [int((n_cat* w['Fluency']) / (max_range+1)) for w in data]
relevance = [int((n_cat* w['Relevance']) / (max_range+1)) for w in data]
structure = [int((n_cat* w['TextStructure']) / (max_range+1)) for w in data]
    
for i, id_ in enumerate(ids):
    for j, row in enumerate(data):
        if row['id'] == id_:
            row_ = copy.copy(row)
            row_['Correctness'] = correctness[i]
            row_['DataCoverage'] = coverage[i]
            row_['Fluency'] = fluency[i]
            row_['Relevance'] = relevance[i]
            row_['TextStructure'] = structure[i]
            data_discretized.append(row_)
            break

data_discretized = sorted(data_discretized, key=lambda x: x['id'])         

Computing the Fleiss' Kappa agreements

In [64]:
assignments = set([(w['submission_id'], w['sample_id']) for w in data_discretized])

correctness = np.zeros((len(assignments), n_cat))
coverage = np.zeros((len(assignments), n_cat))
fluency = np.zeros((len(assignments), n_cat))
relevance = np.zeros((len(assignments), n_cat))
structure = np.zeros((len(assignments), n_cat))

for i, (submission_id, sample_id) in enumerate(assignments):
    fdata = [w for w in data_discretized if w['submission_id'] == submission_id and w['sample_id'] == sample_id]
    
    for rating in fdata:
        correctness[i, rating['Correctness']-1] += 1
        coverage[i, rating['DataCoverage']-1] += 1
        fluency[i, rating['Fluency']-1] += 1
        relevance[i, rating['Relevance']-1] += 1
        structure[i, rating['TextStructure']-1] += 1      
        
pd.DataFrame({"Fleiss' Kappa": {
    'Correctness': fleiss_kappa(correctness),
    'Data Coverage': fleiss_kappa(coverage),
    'Fluency': fleiss_kappa(fluency),
    'Relevance': fleiss_kappa(relevance),
    'Text Structure': fleiss_kappa(structure),
}}).round(3)

,Fleiss' Kappa
Correctness,0.059
Data Coverage,0.068
Fluency,0.058
Relevance,0.041
Text Structure,0.043


# Human Evaluation

Results of the human evaluation for the participating systems according to original ratings of correctness, data coverage, fluency, relevance and text structure.

In [65]:
df = pd.DataFrame(data)

submissions = df.groupby("submission_id")["Correctness", "DataCoverage", "Fluency", "Relevance", "TextStructure"]
submissions.agg([np.mean, np.std]).sort_values(by=('Correctness', 'mean'), ascending=False).round(3)

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:3: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  This is separate from the ipykernel package so we can avoid doing imports until


Correctness         DataCoverage  ... Relevance TextStructure        
                      mean     std         mean  ...       std          mean     std
submission_id                                    ...                                
NUIG-DSI            87.228  17.713       87.582  ...    17.226        86.676  17.769
RALI                87.227  17.804       89.854  ...    16.858        79.298  22.498
REF                 87.082  18.492       88.983  ...    17.527        85.816  19.220
FBConvAI            86.725  18.904       87.459  ...    18.515        87.165  17.857
AmazonAI            86.448  18.478       88.064  ...    16.870        86.067  18.081
bt5                 86.335  19.339       86.689  ...    17.774        85.637  18.681
OSU_Neural_NLG      86.060  18.243       88.345  ...    18.192        86.307  17.361
cuni                85.824  19.973       88.352  ...    16.863        85.227  19.559
DANGNT-SGU          85.105  19.741       88.807  ...    17.687        77.878  22.569
CycleGT             84.949  21.072       85.865  ...    19.196        84.732  19.921
BASELINE2017        84.483  18.537       86.483  ...    15.267        81.633  18.895
BASELINE            84.120  20.425       86.079  ...    19.687        81.258  21.417
TGen                83.994  21.569       83.747  ...    19.581        83.899  20.770
Huawei              76.974  24.564       80.564  ...    23.792        75.092  24.578
ORANGE-NLG          73.341  27.364       77.099  ...    26.335        77.721  23.872
NILC                73.114  28.788       78.176  ...    26.197        76.972  25.465
UPC-POE             73.079  26.008       75.180  ...    24.331        76.185  25.958

[17 rows x 10 columns]

# Human Evaluation (Z-Scores)

Results of the human evaluation for the participating systems according to normalized z-scores for correctness, data coverage, fluency, relevance and text structure.

In [66]:
normdata = []
worker_ids = set([w['worker_id'] for w in data])
for worker_id in worker_ids:
    fdata = [w for w in data if w['worker_id'] == worker_id]
    
    ids = [w['id'] for w in fdata]
    correctness = zscore([w['Correctness'] for w in fdata])
    coverage = zscore([w['DataCoverage'] for w in fdata])
    fluency = zscore([w['Fluency'] for w in fdata])
    relevance = zscore([w['Relevance'] for w in fdata])
    structure = zscore([w['TextStructure'] for w in fdata])
    
    for i, id_ in enumerate(ids):
        for j, row in enumerate(data):
            if row['id'] == id_:
                row_ = copy.copy(row)
                row_['Correctness'] = correctness[i]
                row_['DataCoverage'] = coverage[i]
                row_['Fluency'] = fluency[i]
                row_['Relevance'] = relevance[i]
                row_['TextStructure'] = structure[i]
                normdata.append(row_)
                break
                
df = pd.DataFrame(normdata)

submissions = df.groupby("submission_id")["Correctness", "DataCoverage", "Fluency", "Relevance", "TextStructure"]
submissions.agg([np.mean, np.std]).sort_values(by=('Correctness', 'mean'), ascending=False).round(3)

/usr/local/lib/python3.6/dist-packages/scipy/stats/stats.py:2419: RuntimeWarning: invalid value encountered in true_divide
  return (a - mns) / sstd
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:27: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.


Correctness        DataCoverage  ... Relevance TextStructure       
                      mean    std         mean  ...       std          mean    std
submission_id                                   ...                               
AmazonAI             0.217  0.725        0.207  ...     0.678         0.240  0.738
REF                  0.181  0.762        0.173  ...     0.717         0.162  0.806
OSU_Neural_NLG       0.180  0.726        0.216  ...     0.814         0.238  0.740
NUIG-DSI             0.180  0.770        0.082  ...     0.790         0.221  0.749
RALI                 0.179  0.799        0.269  ...     0.780        -0.211  1.152
bt5                  0.146  0.770        0.060  ...     0.658         0.177  0.739
DANGNT-SGU           0.136  0.754        0.228  ...     0.592        -0.153  1.087
BASELINE             0.131  0.793        0.143  ...     0.809         0.047  0.902
FBConvAI             0.131  0.857        0.077  ...     0.851         0.234  0.693
cuni                 0.114  0.862        0.149  ...     0.755         0.146  0.804
BASELINE2017         0.079  0.779        0.096  ...     0.792         0.028  1.003
TGen                 0.068  0.914       -0.080  ...     0.752         0.095  0.859
CycleGT              0.025  0.950       -0.027  ...     0.857         0.043  0.910
Huawei              -0.238  1.209       -0.182  ...     1.369        -0.302  1.218
ORANGE-NLG          -0.484  1.470       -0.454  ...     1.625        -0.258  1.180
NILC                -0.509  1.423       -0.362  ...     1.440        -0.336  1.324
UPC-POE             -0.536  1.342       -0.597  ...     1.392        -0.371  1.322

[17 rows x 10 columns]

# Statistical Testing

## Wilcoxon rank-sum significant test

In [234]:
def parse(data, normdata):
    correctness, coverage, fluency, relevance, structure = {}, {}, {}, {}, {}
    normcorrectness, normcoverage, normfluency, normrelevance, normstructure = {}, {}, {}, {}, {}

    submission_ids = sorted(list(set([w['submission_id'] for w in data])))
    sample_ids = sorted(list(set([w['sample_id'] for w in data])), key=lambda x: int(x))
    for i, submission_id in enumerate(submission_ids):
        if submission_id not in correctness:
            correctness[submission_id] = []
            coverage[submission_id] = []
            fluency[submission_id] = []
            relevance[submission_id] = []
            structure[submission_id] = []

            normcorrectness[submission_id] = []
            normcoverage[submission_id] = []
            normfluency[submission_id] = []
            normrelevance[submission_id] = []
            normstructure[submission_id] = []
        
        for sample_id in sample_ids:
          fdata = [w for w in data if w['submission_id'] == submission_id and w['sample_id'] == sample_id]
          fnormdata = [w for w in normdata if w['submission_id'] == submission_id and w['sample_id'] == sample_id]

          correctness[submission_id].append(np.mean([w['Correctness'] for w in fdata]))
          coverage[submission_id].append(np.mean([w['DataCoverage'] for w in fdata]))
          fluency[submission_id].append(np.mean([w['Fluency'] for w in fdata]))
          relevance[submission_id].append(np.mean([w['Relevance'] for w in fdata]))
          structure[submission_id].append(np.mean([w['TextStructure'] for w in fdata]))

          # Average the z-scores (setting nans to zeros) of the three turkers for each trial of each system
          normcorrectness[submission_id].append(np.mean(np.nan_to_num([w['Correctness'] for w in fnormdata])))
          normcoverage[submission_id].append(np.mean(np.nan_to_num([w['DataCoverage'] for w in fnormdata])))
          normfluency[submission_id].append(np.mean(np.nan_to_num([w['Fluency'] for w in fnormdata])))
          normrelevance[submission_id].append(np.mean(np.nan_to_num([w['Relevance'] for w in fnormdata])))
          normstructure[submission_id].append(np.mean(np.nan_to_num([w['TextStructure'] for w in fnormdata])))
    return correctness, coverage, fluency, relevance, structure, \
            normcorrectness, normcoverage, normfluency, normrelevance, normstructure
    
def rank_systems(X, raw_X, name):
    submissions = sorted(X.keys(), key=lambda x: np.mean(X[x]), reverse=True)
    ranking = { s:1 for i, s in enumerate(submissions) }

    for i, subA in enumerate(submissions):
        for j, subB in enumerate(submissions[i+1:]):
            s, pvalue = wilcoxon(X[subA], X[subB])
            if pvalue < 0.05:
                ranking[subB] = ranking[subA] + 1
            elif ranking[subB] < ranking[submissions[i+1+j-1]] :
                ranking[subB] = ranking[submissions[i+1+j-1]] 

    ranking_ = {}
    for sub in ranking:
        rank = ranking[sub]
        normmean = np.mean(X[sub])
        mean = np.mean(raw_X[sub])
        ranking_[sub] = { 'Ranking': int(rank), name + ' (Z.)': round(normmean, 3), name: round(mean, 3) }

    return ranking_

correctness, coverage, fluency, relevance, structure, \
      normcorrectness, normcoverage, normfluency, normrelevance, normstructure = parse(data, normdata)


### All Data


In [235]:
pd.DataFrame(rank_systems(normcorrectness, correctness, 'Correctness')).T

,Ranking,Correctness (Z.),Correctness
AmazonAI,1.0,0.216,86.448
REF,1.0,0.181,87.082
OSU_Neural_NLG,1.0,0.180,86.060
NUIG-DSI,1.0,0.180,87.228
RALI,1.0,0.179,87.227
bt5,1.0,0.146,86.335
DANGNT-SGU,1.0,0.136,85.105
BASELINE,2.0,0.131,84.120
FBConvAI,2.0,0.131,86.725
cuni,2.0,0.114,85.824


In [236]:
pd.DataFrame(rank_systems(normcoverage, coverage, 'Coverage')).T

,Ranking,Coverage (Z.),Coverage
RALI,1.0,0.268,89.854
DANGNT-SGU,1.0,0.228,88.807
OSU_Neural_NLG,1.0,0.215,88.345
AmazonAI,1.0,0.207,88.064
REF,2.0,0.173,88.983
cuni,2.0,0.149,88.352
BASELINE,2.0,0.143,86.079
BASELINE2017,2.0,0.096,86.483
NUIG-DSI,2.0,0.082,87.582
FBConvAI,2.0,0.077,87.459


In [237]:
pd.DataFrame(rank_systems(normfluency, fluency, 'Fluency')).T

,Ranking,Fluency (Z.),Fluency
AmazonAI,1.0,0.310,84.116
FBConvAI,1.0,0.243,85.067
OSU_Neural_NLG,1.0,0.229,83.554
NUIG-DSI,1.0,0.200,84.275
REF,2.0,0.181,83.436
cuni,2.0,0.154,82.951
TGen,2.0,0.147,81.678
bt5,2.0,0.130,82.303
CycleGT,3.0,0.039,81.116
BASELINE,3.0,0.022,77.429


In [238]:
pd.DataFrame(rank_systems(normrelevance, relevance, 'Relevance')).T


,Ranking,Relevance (Z.),Relevance
AmazonAI,1.0,0.193,88.449
BASELINE2017,1.0,0.193,88.178
DANGNT-SGU,1.0,0.153,87.436
RALI,1.0,0.153,89.543
cuni,1.0,0.146,89.090
bt5,1.0,0.134,88.184
NUIG-DSI,1.0,0.113,88.536
REF,2.0,0.112,87.687
OSU_Neural_NLG,2.0,0.110,87.019
BASELINE,2.0,0.097,85.912


In [239]:
pd.DataFrame(rank_systems(normstructure, structure, 'Text Structure')).T
    

,Ranking,Text Structure (Z.),Text Structure
AmazonAI,1.0,0.240,86.067
OSU_Neural_NLG,1.0,0.238,86.307
FBConvAI,1.0,0.234,87.165
NUIG-DSI,1.0,0.221,86.676
bt5,1.0,0.177,85.637
REF,1.0,0.162,85.816
cuni,1.0,0.146,85.227
TGen,2.0,0.095,83.899
BASELINE,2.0,0.047,81.258
CycleGT,2.0,0.043,84.732


### Domain Type 1

In [240]:
fnormdata = [w for w in normdata if w['domain'] == 'type1']
fdata = [w for w in data if w['domain'] == 'type1']
correctness, coverage, fluency, relevance, structure, \
  normcorrectness, normcoverage, normfluency, normrelevance, normstructure = parse(fdata, fnormdata)


In [241]:
pd.DataFrame(rank_systems(normcorrectness, correctness, 'Correctness')).T


,Ranking,Correctness (Z.),Correctness
AmazonAI,1.0,0.273,85.951
FBConvAI,1.0,0.256,88.111
bt5,1.0,0.250,88.772
ORANGE-NLG,1.0,0.241,87.389
RALI,1.0,0.221,87.457
REF,1.0,0.194,86.401
cuni,1.0,0.181,85.586
NUIG-DSI,1.0,0.164,87.488
OSU_Neural_NLG,1.0,0.153,85.179
BASELINE,2.0,0.133,82.611


In [242]:
pd.DataFrame(rank_systems(normcoverage, coverage, 'Coverage')).T


,Ranking,Coverage (Z.),Coverage
AmazonAI,1.0,0.269,87.759
cuni,1.0,0.248,88.772
RALI,1.0,0.247,89.179
DANGNT-SGU,1.0,0.203,87.216
REF,1.0,0.201,89.216
BASELINE,1.0,0.166,85.278
NILC,1.0,0.149,89.019
bt5,1.0,0.144,89.309
OSU_Neural_NLG,1.0,0.133,86.969
ORANGE-NLG,1.0,0.115,87.519


In [243]:
pd.DataFrame(rank_systems(normfluency, fluency, 'Fluency')).T


,Ranking,Fluency (Z.),Fluency
FBConvAI,1.0,0.336,86.549
AmazonAI,1.0,0.258,81.222
NUIG-DSI,1.0,0.240,84.957
cuni,1.0,0.202,82.037
bt5,1.0,0.190,83.284
ORANGE-NLG,1.0,0.172,81.864
OSU_Neural_NLG,2.0,0.157,81.309
REF,2.0,0.146,81.654
NILC,2.0,0.106,81.654
Huawei,2.0,0.087,79.321


In [244]:
pd.DataFrame(rank_systems(normrelevance, relevance, 'Relevance')).T


,Ranking,Relevance (Z.),Relevance
cuni,1.0,0.222,88.210
bt5,1.0,0.198,89.056
NUIG-DSI,1.0,0.181,89.432
AmazonAI,1.0,0.174,86.346
REF,1.0,0.149,87.562
NILC,1.0,0.144,88.346
ORANGE-NLG,1.0,0.141,87.488
RALI,1.0,0.121,88.519
TGen,1.0,0.093,87.043
DANGNT-SGU,2.0,0.093,84.969


In [245]:
pd.DataFrame(rank_systems(normstructure, structure, 'Text Structure')).T
    

,Ranking,Text Structure (Z.),Text Structure
bt5,1.0,0.271,87.173
FBConvAI,1.0,0.245,86.673
NUIG-DSI,1.0,0.234,86.432
cuni,1.0,0.219,83.667
AmazonAI,1.0,0.218,83.586
OSU_Neural_NLG,1.0,0.213,84.914
ORANGE-NLG,1.0,0.115,84.358
NILC,1.0,0.105,84.383
REF,1.0,0.100,84.142
UPC-POE,2.0,0.082,82.370


### Domain Type 2

In [246]:
fnormdata = [w for w in normdata if w['domain'] == 'type2']
fdata = [w for w in data if w['domain'] == 'type2']
correctness, coverage, fluency, relevance, structure, \
  normcorrectness, normcoverage, normfluency, normrelevance, normstructure = parse(fdata, fnormdata)


In [247]:
pd.DataFrame(rank_systems(normcorrectness, correctness, 'Correctness')).T


,Ranking,Correctness (Z.),Correctness
OSU_Neural_NLG,1.0,0.220,86.892
BASELINE2017,1.0,0.214,86.180
DANGNT-SGU,1.0,0.210,88.117
RALI,1.0,0.200,87.766
AmazonAI,1.0,0.193,87.730
BASELINE,1.0,0.180,86.559
NUIG-DSI,1.0,0.171,85.649
cuni,1.0,0.160,87.613
CycleGT,1.0,0.132,87.414
TGen,1.0,0.125,84.423


In [248]:
pd.DataFrame(rank_systems(normcoverage, coverage, 'Coverage')).T


,Ranking,Coverage (Z.),Coverage
BASELINE,1.0,0.271,88.874
RALI,1.0,0.266,89.982
DANGNT-SGU,1.0,0.220,89.784
REF,1.0,0.217,91.081
OSU_Neural_NLG,1.0,0.214,89.144
AmazonAI,1.0,0.205,88.532
cuni,1.0,0.190,90.117
BASELINE2017,1.0,0.136,87.333
NUIG-DSI,1.0,0.101,86.847
TGen,1.0,0.089,86.505


In [249]:
pd.DataFrame(rank_systems(normfluency, fluency, 'Fluency')).T


,Ranking,Fluency (Z.),Fluency
AmazonAI,1.0,0.397,86.739
OSU_Neural_NLG,1.0,0.301,85.685
CycleGT,1.0,0.242,85.108
bt5,2.0,0.218,83.820
TGen,2.0,0.218,82.838
FBConvAI,2.0,0.216,84.964
NUIG-DSI,2.0,0.202,83.333
REF,2.0,0.151,85.081
cuni,2.0,0.138,84.009
BASELINE,2.0,0.072,78.838


In [250]:
pd.DataFrame(rank_systems(normrelevance, relevance, 'Relevance')).T


,Ranking,Relevance (Z.),Relevance
BASELINE2017,1.0,0.244,89.207
DANGNT-SGU,1.0,0.231,90.162
AmazonAI,1.0,0.208,89.757
cuni,1.0,0.194,90.676
OSU_Neural_NLG,1.0,0.189,88.189
BASELINE,1.0,0.185,88.865
CycleGT,1.0,0.174,89.243
REF,1.0,0.174,90.514
NUIG-DSI,1.0,0.126,87.441
RALI,1.0,0.124,89.477


In [251]:
pd.DataFrame(rank_systems(normstructure, structure, 'Text Structure')).T


,Ranking,Text Structure (Z.),Text Structure
NUIG-DSI,1.0,0.324,87.360
OSU_Neural_NLG,1.0,0.279,87.505
CycleGT,1.0,0.250,88.748
FBConvAI,1.0,0.211,88.054
AmazonAI,1.0,0.208,87.000
bt5,1.0,0.199,85.450
cuni,1.0,0.193,87.757
REF,1.0,0.166,88.117
TGen,1.0,0.163,84.234
BASELINE,1.0,0.081,83.378


### Domain Type 3

In [252]:
fnormdata = [w for w in normdata if w['domain'] == 'type3']
fdata = [w for w in data if w['domain'] == 'type3']
correctness, coverage, fluency, relevance, structure, \
  normcorrectness, normcoverage, normfluency, normrelevance, normstructure = parse(fdata, fnormdata)


In [253]:
pd.DataFrame(rank_systems(normcorrectness, correctness, 'Correctness')).T


,Ranking,Correctness (Z.),Correctness
REF,1.0,0.208,86.828
NUIG-DSI,1.0,0.193,87.739
AmazonAI,1.0,0.191,86.211
OSU_Neural_NLG,1.0,0.181,86.253
DANGNT-SGU,1.0,0.155,85.709
RALI,1.0,0.144,86.854
bt5,1.0,0.129,85.245
BASELINE,1.0,0.109,84.019
FBConvAI,1.0,0.090,85.667
BASELINE2017,2.0,0.067,84.946


In [254]:
pd.DataFrame(rank_systems(normcoverage, coverage, 'Coverage')).T


,Ranking,Coverage (Z.),Coverage
RALI,1.0,0.282,90.218
OSU_Neural_NLG,1.0,0.267,88.858
DANGNT-SGU,1.0,0.248,89.379
AmazonAI,1.0,0.169,88.054
REF,2.0,0.137,87.946
BASELINE2017,2.0,0.114,86.870
BASELINE,2.0,0.075,85.387
CycleGT,2.0,0.073,88.295
cuni,2.0,0.070,87.341
NUIG-DSI,2.0,0.058,88.050


In [255]:
pd.DataFrame(rank_systems(normfluency, fluency, 'Fluency')).T


,Ranking,Fluency (Z.),Fluency
AmazonAI,1.0,0.305,84.797
OSU_Neural_NLG,1.0,0.242,84.042
REF,1.0,0.215,83.843
FBConvAI,1.0,0.196,84.192
NUIG-DSI,1.0,0.174,84.253
TGen,2.0,0.155,81.602
cuni,2.0,0.130,83.069
bt5,2.0,0.055,81.050
CycleGT,2.0,0.034,80.966
BASELINE,3.0,-0.022,78.100


In [256]:
pd.DataFrame(rank_systems(normrelevance, relevance, 'Relevance')).T


,Ranking,Relevance (Z.),Relevance
BASELINE2017,1.0,0.253,89.100
AmazonAI,1.0,0.198,89.199
RALI,1.0,0.185,90.207
DANGNT-SGU,1.0,0.158,87.808
bt5,2.0,0.123,87.644
BASELINE,2.0,0.099,86.096
OSU_Neural_NLG,2.0,0.091,87.475
TGen,2.0,0.086,87.107
CycleGT,2.0,0.086,89.456
cuni,2.0,0.079,88.962


In [257]:
pd.DataFrame(rank_systems(normstructure, structure, 'Text Structure')).T


,Ranking,Text Structure (Z.),Text Structure
AmazonAI,1.0,0.267,87.211
FBConvAI,1.0,0.236,87.092
OSU_Neural_NLG,1.0,0.235,86.663
REF,1.0,0.199,85.877
NUIG-DSI,1.0,0.169,86.536
bt5,1.0,0.109,84.762
TGen,1.0,0.090,84.054
cuni,2.0,0.080,85.119
CycleGT,2.0,0.062,85.716
BASELINE2017,2.0,0.055,82.490
